# 數位控制系統第四章：開迴路離散時間系統（教學版 Notebook）

本 Notebook 是 `chp4.md` 教材的教學版，額外補充：

- 每個第一次出現的 MATLAB / Octave 函數（`tf`, `c2d`, `tfdata`, `ssdata`, `dcgain`, `filter`, `expm`, ...）的逐步解說
- 多項式係數向量、`expm` vs `exp`、`filter` 對齊等初學者容易卡住的語法
- 公式 → 程式碼的逐項對照（每段程式都標註對應的教材式號與範例編號）
- 七個可直接執行的實驗，親手驗證本章每一條公式

建議搭配 `chp4.md`（完整理論、推導與符號定義）與 `chp4.m`（精簡可執行版）一起閱讀。

> **本章要解決的問題**：第 3 章證明了「理想取樣器沒有轉移函數」，因此離散系統的方塊圖不能像連續系統那樣直接相乘化簡。本章推導出**脈衝轉移函數** $G(z)$ 來解決它。

---


## 🔧 環境設定

> **需要 Control System Toolbox**（Octave 為 `control` 套件）。本章大量使用 `tf`、`c2d`、`ssdata`、`dcgain`，沒有這個套件完全無法執行。
>
> `if exist('OCTAVE_VERSION','builtin')` 讓同一段程式在 **MATLAB 與 Octave 都能直接跑**——MATLAB 會自動跳過整段。


In [ ]:
%plot --format svg

if exist('OCTAVE_VERSION', 'builtin')
    warning('off', 'Octave:gnuplot-graphics');
    warning('off', 'Octave:fltk-graphics');
    graphics_toolkit('gnuplot');
    pkg load control;
end
clear; clc;

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

T = 0.1;                  % 取樣週期 (s)
printf('取樣週期 T = %g s\n', T);

## 📖 全章符號總表

| 符號 | 型別 | 意義 |
|---|---|---|
| $E(z)$ | 複變函數 | 數列 $\{e(k)\}$ 的 **z 轉換** |
| $E^*(s)$ | 複變函數 | $e(t)$ 的**星號轉換**（第 3 章式 3-3） |
| $G_p(s)$ | 複變函數 | 受控體轉移函數 |
| $G(s)$ | 複變函數 | **保持器 × 受控體**的合併轉移函數 |
| $G(z)$ | 複變函數 | **脈衝轉移函數** |
| $D(z)$ | 複變函數 | 數位濾波器轉移函數 |
| $\Delta$ | 純量 | 延遲佔取樣週期的比例，$0<\Delta<1$ |
| $m$ | 純量 | 修正 z 轉換參數，$m=1-\Delta$ |
| $E(z,m)$ | 複變函數 | **修正 z 轉換** |
| $t_0$ | 純量 | 理想時間延遲（秒），$t_0=kT+\Delta T$ |
| $A_c,B_c,C_c,D_c$ | 矩陣 | **連續**時間狀態空間矩陣（下標 c） |
| $A,B,C,D$ | 矩陣 | **離散**時間狀態空間矩陣（無下標） |
| $\Phi_c(t)$ | $n\times n$ | 狀態轉移矩陣 $=e^{A_ct}$ |

## 📖 本章會用到的 Octave 語法小抄

| 語法 | 意義 | 本章用途 |
|---|---|---|
| `tf(num, den)` | 建立**連續**轉移函數 | $G_p(s)$ |
| `tf(num, den, T)` | 第三引數 `T` → **離散**轉移函數 | $D(z)$ |
| `c2d(sys, T, 'zoh')` | 連續轉離散 | **`'zoh'` 就是 $\dfrac{1-e^{-Ts}}{s}$** |
| `tfdata(sys, 'v')` | 取出分子分母**係數向量**（`'v'` = vector） | 比對教材公式 |
| `ssdata(sys)` | 取出 $A,B,C,D$ 四個矩陣 | 例 4.13 |
| `dcgain(sys)` | 直流增益（離散代 $z=1$、連續代 $s=0$） | 式 4-14 檢查 |
| `conv(a, b)` | 多項式**相乘** | $(z-1)(z-e^{-T})$ |
| `polyval(p, x)` | 多項式在 `x` 求值 | 手動代 $z=1$ |
| **`expm(M)`** | **矩陣指數** $e^M$ | $\Phi_c(T)=e^{A_cT}$ |
| `exp(x)` | **逐元素**指數 | $e^{-T}$（純量時才用） |
| **`A^k`** | **矩陣次方** | 級數展開 $A_c^k$ |
| `A.^k` | 逐元素次方 | 本章**不要**用在 $A_c^k$ |
| `factorial(k)` | $k!$ | 級數展開的分母 |
| `filter(b, a, x)` | 依差分方程式濾波 | 取**反 z 轉換序列** |
| `step(sys, t)` | 階躍響應 | 各範例的時間響應 |
| `stairs(x, y)` | 階梯圖 | 畫離散響應 |

> **⚠️ 本章最致命的錯誤：`expm` 與 `exp` 搞混。**
> $e^{A_cT}$ 是**矩陣指數**（定義為級數 $I+A_cT+\frac{(A_cT)^2}{2!}+\cdots$），必須用 `expm(Ac*T)`。
> 若寫成 `exp(Ac*T)`，Octave 會對矩陣的**每個元素**分別取指數——**答案完全錯誤，而且不會報錯**。

---


## 二、$E(z)$ 與 $E^*(s)$ 的關係 (4.2)

### 核心關係（式 4-3）

第 2 章的 z 轉換與第 3 章的星號轉換擺在一起：

$$E(z)=e(0)+e(1)z^{-1}+e(2)z^{-2}+\cdots$$

$$E^*(s)=e(0)+e(T)e^{-Ts}+e(2T)e^{-2Ts}+\cdots$$

**若 $e(k)=e(kT)$ 且令 $e^{sT}=z$，兩者完全相同**：

$$\boxed{E(z)=E^*(s)\Big|_{e^{sT}=z}}$$

> **💡 這句話的份量**：**z 轉換是拉氏轉換的一個特例**。第 2 章對 z 轉換證明的所有定理，全部自動適用於星號轉換——這也是為什麼教科書不另外列星號轉換表。
>
> 第 3 章實驗 3 已經預告過這件事：單位步階的 $E^*(s)=\dfrac{1}{1-e^{-Ts}}$，代入 $z=e^{Ts}$ 就是 $\dfrac{z}{z-1}$。

### 為什麼要改用 $E(z)$

| | $E^*(s)$ | $E(z)$ |
|---|---|---|
| 極點數 | **無限多**（第 3 章性質 2） | **有限個** |
| 極零點分析 | 幾乎無法使用 | 直接可用 |

**範例 4.1**：$E(s)=\dfrac{1}{(s+1)(s+2)}$ 的 $E^*(s)$ 在 $s$ 平面有無限多極零點，但 $E(z)=\dfrac{z(e^{-T}-e^{-2T})}{(z-e^{-T})(z-e^{-2T})}$ 只有一個零點、兩個極點。

---


## 三、脈衝轉移函數 (4.3)

### 推導的關鍵一步

$$C(s)=G(s)E^*(s)\ \xrightarrow{\ \text{式 3-11}\ }\ C^*(s)=\frac1T\sum_{n}G(s+jn\omega_s)E^*(s+jn\omega_s)$$

**用第 3 章性質 1 的週期性** $E^*(s+jn\omega_s)=E^*(s)$，把 $E^*(s)$ 提到求和外：

$$C^*(s)=E^*(s)\cdot\frac1T\sum_n G(s+jn\omega_s)=E^*(s)G^*(s)$$

代入 $z=e^{Ts}$：

$$\boxed{C(z)=G(z)E(z)}$$

> **💡 為什麼這是本章的核心**：第 3 章說「取樣器沒有轉移函數」，看起來離散系統無法用轉移函數分析。但這個推導證明——**只要在取樣瞬間看，輸入輸出之間確實存在轉移函數關係**。讓一切成立的關鍵，正是星號轉換的**週期性**。

### ⚠️ 根本限制

> **脈衝轉移函數只給出「取樣瞬間」的輸出，對取樣瞬間之間的 $c(t)$ 一無所知。**

實務作法：把取樣頻率選得夠高，讓取樣點的響應能反映取樣點之間的響應；真的需要完整響應時用**模擬**。（第 4.5 節的修正 z 轉換也能看到取樣點之間。）

### 這段程式在做什麼

範例 4.2、4.3：$G_p(s)=\dfrac{1}{s+1}$，求 $G(z)=\mathcal{Z}\left[\dfrac{1-e^{-Ts}}{s(s+1)}\right]=\dfrac{1-e^{-T}}{z-e^{-T}}$。

**數學 ↔ MATLAB 對照**：

| 數學 | 程式 | 說明 |
|---|---|---|
| $G_p(s)=\dfrac{1}{s+1}$ | `tf(1, [1 1])` | 分子 $[1]$；分母 $1\cdot s+1\to[1\ 1]$ |
| $\dfrac{1-e^{-Ts}}{s}G_p(s)$ 再取 z 轉換 | `c2d(Gp, T, 'zoh')` | **`'zoh'` 這個字串就是那個保持器** |
| 取出係數 | `[nz,dz] = tfdata(Gz,'v')` | `'v'` 代表回傳向量而非 cell |

> **`tf` 的多項式係數怎麼寫？** 由**高次到低次**排列。例如 $s+1\to$ `[1 1]`、$s^2+3s+2\to$ `[1 3 2]`、$2z-1\to$ `[2 -1]`。


In [ ]:
%% 實驗 1：脈衝轉移函數（例 4.2、4.3）
Gp = tf(1, [1 1]);              % Gp(s) = 1/(s+1)
Gz = c2d(Gp, T, 'zoh');         % G(z) = Z[(1-e^{-Ts})/s * Gp(s)]
[nz, dz] = tfdata(Gz, 'v');     % 取出分子、分母係數向量

printf('c2d 給的 G(z) = (%.6f z + %.6f) / (z %+.6f)\n', nz(1), nz(2), dz(2));
printf('教材公式      = %.6f / (z - %.6f)\n', 1-exp(-T), exp(-T));
printf('分子常數項差異 = %.2e，分母差異 = %.2e\n\n', ...
       abs(nz(2)-(1-exp(-T))), abs(dz(2)+exp(-T)));

% 階躍響應與教材 c(kT) = 1 - e^{-kT} 比對
n = 0:8;
[y1, ~] = step(Gz, 0:T:8*T);
c1_theory = 1 - exp(-n*T);
printf('  n     模擬 c(nT)    教材 1-e^{-nT}   差\n');
for k = 1:numel(n)
    printf('  %-5d %-13.6f %-16.6f %.2e\n', n(k), y1(k), c1_theory(k), abs(y1(k)-c1_theory(k)));
end

### 結果解讀

`c2d` 算出的 $G(z)$ 與教材公式 $\dfrac{1-e^{-T}}{z-e^{-T}}$ 完全一致（差異 0），階躍響應也精確等於 $c(kT)=1-e^{-kT}$。

> **教材的重要提醒**：本例輸入是單位步階，而**取樣器加 ZOH 對步階訊號的重建是完全精確的**（步階進去、步階出來）。因此這個系統的響應其實就等於連續系統 $\frac{1}{s+1}$ 的步階響應。
>
> **但反過來不成立**：由 $c(t)$ 可以代 $t=kT$ 得 $c(kT)$；**但由 z 轉換得到的 $c(kT)$，一般不能把 $kT$ 換成 $t$ 就當成 $c(t)$**。

---


## 直流增益檢查 (式 4-14)

### 公式

由**終值定理**，單位步階輸入下：

$$c_{ss}=\lim_{z\to1}(z-1)C(z)=\lim_{z\to1}(z-1)G(z)\frac{z}{z-1}=G(1)$$

又因為對常數輸入，取樣器加 ZOH 的增益是 1：

$$\boxed{\text{dc gain}=\lim_{z\to1}G(z)=\lim_{s\to0}G_p(s)}$$

> **💡 這是本章最實用的檢查工具**：兩邊都很容易算。你算出 $G(z)$ 之後代 $z=1$，跟 $G_p(s)$ 代 $s=0$ 比一比——**不相等就是算錯了**。

**三種算法**（下一格全部跑一次）：

| 數學 | 程式 |
|---|---|
| 手動代 $z=1$ | `polyval(nz,1)/polyval(dz,1)` |
| 離散 dc 增益 | `dcgain(Gz)` |
| 連續 dc 增益 | `dcgain(Gp)` |


In [ ]:
%% 實驗 2：直流增益檢查（式 4-14）
printf('G(1)            = %.8f   （手動代 z=1）\n', polyval(nz,1)/polyval(dz,1));
printf('dcgain(Gz)      = %.8f   （離散系統，代 z=1）\n', dcgain(Gz));
printf('lim Gp(s), s->0 = %.8f   （連續系統，代 s=0）\n', dcgain(Gp));
printf('=> 三者相同，G(z) 的計算正確\n');

### 結果解讀

三種算法都得到 1.00000000。

**養成習慣**：每次算完 $G(z)$，花三秒做這個檢查。它抓得到大部分的代數錯誤——特別是忘記乘保持器、或是 z 轉換查錯表。

---


## 四、⚠️ 全章最容易犯的錯 (式 4-19)

### 兩種串聯組態

**組態 (a)：兩受控體之間「有」取樣器**

```text
      T                    T
E ───/───► G1(s) ───A───/───► G2(s) ───► C
```

$$C(z)=G_1(z)G_2(z)E(z)$$

**組態 (b)：兩受控體之間「沒有」取樣器**

```text
      T
E ───/───► G1(s) ──► G2(s) ───► C
```

$$C(z)=\overline{G_1G_2}(z)E(z),\qquad \overline{G_1G_2}(z)=\mathcal{Z}[G_1(s)G_2(s)]$$

**橫線的意思**：必須**先在 $s$ 域相乘，然後才取 z 轉換**。

### 結論

$$\boxed{\overline{G_1G_2}(z)\neq G_1(z)G_2(z)}$$

> **為什麼？** 因為兩者是**物理上不同的系統**：組態 (a) 中間多了一個取樣器加保持器，訊號在那裡被「階梯化」了一次。**硬體不同，答案當然不同。**

### 程式怎麼寫

| 數學 | 程式 | 關鍵差別 |
|---|---|---|
| $G_1(z)G_2(z)$ | `c2d(G1s,T,'zoh') * c2d(G2s,T,'zoh')` | **各自離散化後才相乘** |
| $\overline{G_1G_2}(z)$ | `c2d(G1s*G2s, T, 'zoh')` | **先在 $s$ 域相乘，再離散化** |

注意這兩行的**括號位置**——差別就在「乘法在離散化之前還是之後」。


In [ ]:
%% 實驗 3：G1G2(z) 不等於 G1(z)*G2(z)（式 4-19）
G1s = tf(1, [1 1]);            % G1(s) = 1/(s+1)
G2s = tf(1, [1 2]);            % G2(s) = 1/(s+2)

G1z = c2d(G1s, T, 'zoh');
G2z = c2d(G2s, T, 'zoh');
prod_sep = G1z * G2z;                % 組態(a)：各自離散化後相乘
G12bar   = c2d(G1s*G2s, T, 'zoh');   % 組態(b)：先在 s 域相乘再離散化

printf('兩者的 dcgain：%.6f  vs  %.6f  （穩態相同！）\n\n', ...
       dcgain(prod_sep), dcgain(G12bar));

[ya, ~] = step(prod_sep, 0:T:6*T);
[yb, ~] = step(G12bar,   0:T:6*T);
printf('但暫態完全不同：\n');
printf('  n     組態(a) G1(z)G2(z)   組態(b) G1G2bar(z)   差\n');
for k = 1:7
    printf('  %-5d %-21.6f %-21.6f %.6f\n', k-1, ya(k), yb(k), abs(ya(k)-yb(k)));
end

figure('Position', [50 50 800 380]);
stairs(0:T:6*T, ya, 'b-', 'LineWidth', 2); hold on;
stairs(0:T:6*T, yb, 'r--', 'LineWidth', 2); grid on;
title('式 4-19：中間有無取樣器，響應完全不同');
xlabel('時間 t (秒)'); ylabel('c(kT)');
legend('組態(a)：G_1(z)G_2(z)（中間有取樣器）', ...
       '組態(b)：G_1G_2(z)（中間無取樣器）', 'Location', 'southeast');

### 結果解讀

**兩條曲線的穩態相同（dcgain 都是 0.5），但暫態明顯不同。**

特別注意 $n=1$：組態 (a) 還是 0，組態 (b) 已經是 0.0045。**組態 (a) 多了一步的延遲**——因為訊號要多經過一次「取樣 → 保持」，而 ZOH 本身就等效於 $T/2$ 的延遲（第 3 章實驗 9）。

> **實務教訓**：畫方塊圖時，**取樣器的位置絕對不能隨便挪**。看到兩個方塊串在一起，第一件事是問「它們中間有沒有取樣器？」——答案決定你要用哪一個公式。

### 組態 (c)：連轉移函數都寫不出來

還有第三種情形：輸入**先經過連續部分才被取樣**。

```text
E ───► G1(s) ───A───/───► G2(s) ───► C
                    T
```

此時 $C(z)=G_2(z)\overline{G_1E}(z)$，**無法把 $E(z)$ 分離出來**——因為

$$a(t)=\int_0^t g_1(t-\tau)e(\tau)\,d\tau$$

$a(t)$ 取決於 $e(t)$ **過去的所有值**，而不只是取樣瞬間的值。

**一般結論**：若輸入先施加到系統的連續部分再被取樣，輸出的 z 轉換無法表示成輸入 z 轉換的函數。（教材補充：這類系統在後續分析設計中不會造成特別困難。）

---


## 五、含數位濾波器的開迴路系統 (4.4)

### 系統架構與公式

```text
        ┌─────┐  {e(kT)}  ┌──────┐  {m(kT)}  ┌─────┐  m(t)  ┌───────┐  c(t)
e(t) ──►│ A/D ├──────────►│ D(z) ├──────────►│ D/A ├───────►│ Gp(s) ├──────►
        └─────┘           └──────┘           └─────┘        └───────┘
```

**關鍵物理事實**：D/A 轉換器通常帶有**輸出資料保持暫存器**，使它具有**零階保持器的特性**。因此：

$$\boxed{C(z)=\mathcal{Z}\left[G_p(s)\frac{1-e^{-Ts}}{s}\right]D(z)E(z)=G(z)D(z)E(z)}$$

> **⚠️ 模型細節**：實體電腦處理的是**數值** $\{e(kT)\}$，但數學模型處理的是**權重為 $\{e(kT)\}$ 的脈衝序列**。因此完整模型必須是「**理想取樣器 + $D(z)$ + 零階保持器**」三者的組合。

### 範例 4.4

差分方程式 $m(kT)=2e(kT)-e[(k-1)T]$，因此

$$D(z)=\frac{M(z)}{E(z)}=2-z^{-1}=\frac{2z-1}{z}$$

$$C(z)=D(z)G(z)E(z)=\frac{(2z-1)(1-e^{-T})}{(z-1)(z-e^{-T})}$$

$$c(nT)=1+(e^{T}-2)e^{-nT}\ (n\ge1),\qquad c(0)=0$$

> **快速檢查技巧**：$c(0)=0$ 一眼就看得出來——**因為 $C(z)$ 的分子次數（1）低於分母次數（2）**。

### 程式怎麼寫 $D(z)$

$$D(z)=\frac{2z-1}{z}\quad\longrightarrow\quad \texttt{tf([2 -1], [1 0], T)}$$

| 部分 | 係數向量 | 代表 |
|---|---|---|
| 分子 $2z-1$ | `[2 -1]` | $2\cdot z^1+(-1)\cdot z^0$ |
| 分母 $z$ | `[1 0]` | $1\cdot z^1+0\cdot z^0$ |
| 第三引數 | `T` | **宣告這是離散系統**（不寫會被當成連續！） |


In [ ]:
%% 實驗 4：含數位濾波器（例 4.4）
Dz = tf([2 -1], [1 0], T);     % D(z) = (2z-1)/z，第三引數 T 宣告為離散
Cz = Dz * Gz;                  % C(z) = D(z)G(z)

[y4, ~] = step(Cz, 0:T:6*T);
n4 = 0:6;
c4_theory = 1 + (exp(T)-2)*exp(-n4*T);   % 教材：c(nT)=1+(e^T-2)e^{-nT}, n>=1
c4_theory(1) = 0;                         % 教材：c(0)=0

printf('  n     模擬 c(nT)    教材公式        差\n');
for k = 1:numel(n4)
    printf('  %-5d %-13.6f %-15.6f %.2e\n', n4(k), y4(k), c4_theory(k), abs(y4(k)-c4_theory(k)));
end

printf('\n三種驗證方式：\n');
printf('  (1) 終值定理  lim c(nT)      = %.6f\n', dcgain(Cz));
printf('  (2) 式(4-14)  D(1)*Gp(0)     = %g * %g = %g\n', ...
       dcgain(Dz), dcgain(Gp), dcgain(Dz)*dcgain(Gp));
printf('  (3) 追蹤訊號  m 穩態 = 2-1  = 1，代回例 4.3 -> 輸出 1\n');
printf('  => 三者一致，教材為 1\n');

### 結果解讀

模擬結果與教材公式完全吻合（誤差 $10^{-17}$ 等級）。

**教材示範的三重驗證值得學起來**——它們從三個完全不同的角度得到同一個答案：

| 方法 | 角度 |
|---|---|
| 終值定理 | z 域的極限運算 |
| 直流增益相乘 | 各方塊增益的乘積 |
| 追蹤訊號 | 物理上「濾波器輸出穩態是多少」 |

**做完任何離散系統分析，至少要用其中一種檢查。**

---


## 六、修正 z 轉換與時間延遲 (4.5、4.6)

### 為什麼需要修正 z 轉換

前面的方法**不適用於含理想時間延遲的系統**。特別是延遲量**不是取樣週期整數倍**的時候。

### 定義（式 4-26、4-27）

**延遲 z 轉換**（延遲 $\Delta T$，但**取樣本身不延遲**）：

$$E(z,\Delta)=\mathcal{Z}[e(t-\Delta T)u(t-\Delta T)]=\mathcal{Z}[E(s)e^{-\Delta Ts}]$$

**修正 z 轉換**（令 $m=1-\Delta$）：

$$E(z,m)=E(z,\Delta)\Big|_{\Delta=1-m}=e(mT)z^{-1}+e[(1+m)T]z^{-2}+\cdots$$

> **💡 $m$ 的物理意義**：$m$ 是「**在每個取樣區間內往前推進的比例**」。$E(z,m)$ 給出訊號在 $t=mT,(1+m)T,(2+m)T,\dots$ 這些**取樣點之間的位置**上的值。
>
> **這正好補足第 4.3 節指出的限制**——普通 z 轉換只看得到取樣瞬間，**修正 z 轉換讓你看到取樣點之間**。

**兩個端點**：$E(z,1)=E(z)-e(0)$（無延遲），$E(z,0)=z^{-1}E(z)$（延遲一整個週期）。

### 含時間延遲的核心公式（式 4-38、4-39）

把延遲拆成「整數倍」加「零頭」：

$$t_0=kT+\Delta T,\qquad 0<\Delta<1$$

$$\boxed{C(z)=z^{-k}\,G(z,m)\,E(z),\qquad m=1-\Delta}$$

| 部分 | 用什麼處理 | 變成 |
|---|---|---|
| 整數部分 $kT$ | 平移定理 | $z^{-k}$ |
| 零頭 $\Delta T$ | 修正 z 轉換 | $G(z,m)$ |

### 範例 4.8

$t_0=0.4T$（所以 $k=0$、$\Delta=0.4$、$m=0.6$），$G(s)=\dfrac{1-e^{-Ts}}{s(s+1)}$，單位步階輸入：

$$C(z)=\frac{z(1-e^{-0.6T})+e^{-0.6T}-e^{-T}}{(z-1)(z-e^{-T})}$$

反 z 轉換應該得到：$c(nT)=1-e^{-(n-0.4)T}$（也就是範例 4.3 的響應**整體延後 $0.4T$**）。

### ⚠️ 程式實作：怎麼取反 z 轉換

Octave 沒有直接的符號反 z 轉換。有兩條路：

| 方法 | 寫法 | 注意事項 |
|---|---|---|
| `filter` | `filter(num, den, [1 zeros(1,N-1)])` | **分子要補零對齊分母次數** |
| `impulse` | `impulse(sys, t)` | **Octave 對離散系統會乘上 $1/T$**，要再乘回 `T` |

本 Notebook 用 `filter`，因為它沒有縮放慣例的問題。

**為什麼分子要補零？** `filter(b,a,x)` 把 `b`、`a` 當成 $z^{-1}$ 的次冪係數：

$$H(z^{-1})=\frac{b_0+b_1z^{-1}+\cdots}{a_0+a_1z^{-1}+\cdots}$$

我們的分子 `[1-e^{-mT}, e^{-mT}-e^{-T}]` 代表的是 $(\cdot)z^1+(\cdot)z^0$（**正次冪**）。直接傳進去，`filter` 會當成 $z^0$ 與 $z^{-1}$——**整個序列提前一步**。前面補一個 0 就對齊了。

| 數學 | 程式 |
|---|---|
| $(z-1)(z-e^{-T})$ | `conv([1 -1], [1 -exp(-T)])` |
| 分子補零 | `[0, num]` |
| 單位樣本序列 | `[1, zeros(1,N-1)]` |


In [ ]:
%% 實驗 5：修正 z 轉換與時間延遲（例 4.8，t0 = 0.4T）
mT   = 0.6*T;                                % m = 1-0.4 = 0.6
num5 = [1-exp(-mT), exp(-mT)-exp(-T)];       % 正次冪 z 的係數
den5 = conv([1 -1], [1 -exp(-T)]);           % (z-1)(z-e^{-T})，conv = 多項式相乘

printf('C(z) 分子 = [%.6f  %.6f]\n', num5);
printf('C(z) 分母 = [%.6f  %.6f  %.6f]\n\n', den5);

% 反 z 轉換：分子補零對齊分母次數，再用 filter
num5_pad = [0, num5];
N5 = 7;
c5 = filter(num5_pad, den5, [1, zeros(1, N5-1)]);

printf('  n     反z轉換 c(nT)   教材 1-e^{-(n-0.4)T}   差\n');
for k = 1:N5
    nn = k-1;
    if nn == 0, th = 0; else th = 1 - exp(-(nn-0.4)*T); end
    printf('  %-5d %-16.6f %-22.6f %.2e\n', nn, c5(k), th, abs(c5(k)-th));
end

figure('Position', [50 50 800 380]);
stairs(0:T:6*T, y1(1:7), 'b-', 'LineWidth', 2); hold on;
stairs(0:T:6*T, c5, 'r--', 'LineWidth', 2); grid on;
title('例 4.8：理想時間延遲 t_0 = 0.4T 的效果');
xlabel('時間 t (秒)'); ylabel('c(kT)');
legend('無延遲（例 4.3）', '延遲 0.4T（例 4.8）', 'Location', 'southeast');

### 結果解讀

反 z 轉換的序列與 $1-e^{-(n-0.4)T}$ 完全吻合——**驗證了修正 z 轉換確實正確處理了非整數倍的延遲**。

圖上兩條階梯的差距，就是那 $0.4T$ 的延遲造成的。**延遲讓響應整體向右平移，這在控制系統中是穩定度的殺手**（第 3 章實驗 9 已經看過 ZOH 本身就貢獻了 $T/2$）。

### 實務應用：電腦的運算時間

修正 z 轉換也用來處理**數位電腦運算時間不可忽略**的情形（式 4-41～4-44）。

**建模方式**：把數位控制器模型化為「**無延遲的控制器 + 一個 $t_0$ 秒的理想時間延遲**」串聯。於是：

$$C(z)=z^{-k}G(z,m)D(z)E(z),\qquad t_0=kT+\Delta T,\ m=1-\Delta$$

**範例 4.9** 就是範例 4.4 加上 1 ms 的濾波器運算延遲（$T=0.05$ s）。

### 非同步取樣 (4.7)

若系統中**兩個取樣器不同步**（一個在 $0,T,2T,\dots$，另一個在 $hT,T+hT,\dots$），也用修正 z 轉換處理：

$$C(z)=z\,E(z)\,G_1(z,m)\Big|_{m=h}\,G_2(z,m)\Big|_{m=1-h}$$

> **💡 模型的巧妙之處**：把「偏移取樣」改寫成「延遲 $(1-h)T$ → 同步取樣 → 超前 $T$ → 延遲 $hT$」，**總延遲恰好為零**（$-(1-h)T+T-hT=0$），但裡面的取樣器全部同步了——問題就化簡成組態 (a) 的形式。

---


## 七、離散時間狀態方程式 (4.8 ~ 4.10)

### 為什麼要換一套做法

轉移函數法有**兩個缺點**：

| 缺點 | 說明 |
|---|---|
| **失去自然狀態** | 用轉移函數法很難把「速度」這種**物理變數**選為狀態——系統自然的狀態被丟掉了 |
| **高階系統困難** | 高階系統推導脈衝轉移函數本身就很困難 |

**第 4.10 節的做法直接從連續時間狀態方程式出發，連續模型的狀態就直接成為離散模型的狀態。**

### 核心公式（式 4-69、4-71、4-70）

由連續解 $\mathbf{v}(t)=\Phi_c(t-t_0)\mathbf{v}(t_0)+\int_{t_0}^t\Phi_c(t-\tau)B_c\mathbf{u}(\tau)d\tau$，在 $t=kT+T$、$t_0=kT$ 求值：

$$\boxed{A=\Phi_c(T)=e^{A_cT},\qquad B=\left[\int_0^T\Phi_c(\nu)\,d\nu\right]B_c,\qquad C=C_c,\quad D=D_c}$$

> **⚠️ 為什麼 $\mathbf{u}$ 可以提到積分外面？** 因為在 $kT\le t<kT+T$ 內 $\mathbf{u}(t)=\mathbf{u}(kT)$ 是常數。**教材特別強調：這個推導只有在 $\mathbf{u}(t)$ 是零階保持器的輸出時才成立。**

> **💡 離散化只改變 $A$ 和 $B$，不改變 $C$ 和 $D$**——因為 $C$、$D$ 描述量測關係，與你多久看一次無關。

### 級數展開（式 4-72、4-74）

$$\Phi_c(T)=I+A_cT+\frac{A_c^2T^2}{2!}+\cdots,\qquad B=\left[IT+\frac{A_cT^2}{2!}+\frac{A_c^2T^3}{3!}+\cdots\right]B_c$$

> **教材說**：三位有效數字需 **3 項**，六位有效數字需 **5 項**。（實驗 7 會驗證。）
>
> **一個實作巧思**：兩個級數的通項只差一個 $\frac{T}{k+1}$ 因子，所以**算 $\Phi_c(T)$ 的程式稍加擴充就能同時算出積分項**。

### 範例 4.13：$G_p(s)=\dfrac{10}{s(s+1)}$，$T=0.1$ s

$$A_c=\begin{bmatrix}0&1\\0&-1\end{bmatrix},\quad B_c=\begin{bmatrix}0\\10\end{bmatrix},\quad C_c=\begin{bmatrix}1&0\end{bmatrix},\quad D_c=0$$

手算 $\Phi_c(t)=\begin{bmatrix}1&1-e^{-t}\\0&e^{-t}\end{bmatrix}$，代入 $T=0.1$：

$$A=\begin{bmatrix}1&0.0952\\0&0.905\end{bmatrix},\qquad B=\begin{bmatrix}0.0484\\0.952\end{bmatrix}$$

### ⚠️ 教材範例 4.14 的程式在現行環境無法執行

教材給的程式是：

```matlab
[A,B] = c2d(Ac,Bc,T)
[numz,denz] = ss2tf(A,B,C,D)
```

**實測結果**：

```text
[A,B] = c2d(Ac,Bc,T)   ->  error: 'c2d' undefined
ss2tf(A,B,C,D)         ->  error: 'ss2tf' undefined
```

**原因**：

1. `c2d(Ac,Bc,T)` 這種**傳四個裸矩陣**的呼叫方式，是舊版 Control System Toolbox 的語法。現行 `c2d` **只接受 LTI 物件**。
2. Octave 的 `control` 套件**沒有 `ss2tf`**。

**依課程規範不逕自改寫教材，而是保留原寫法並提供可執行的修正版**（數學內容完全相同）：

| 教材寫法 | 修正版 | 說明 |
|---|---|---|
| `[A,B] = c2d(Ac,Bc,T)` | `sys_d = c2d(ss(Ac,Bc,Cc,Dc), T, 'zoh')` | 先用 `ss()` 打包 |
| `ss2tf(Ac,Bc,Cc,Dc)` | `tf(sys_c)` | 直接做物件轉換 |
| `ss2tf(A,B,C,D)` | `tf(sys_d)` | 同上 |


In [ ]:
%% 實驗 6：離散狀態方程式與脈衝轉移函數（例 4.13、4.14）
Ac = [0 1; 0 -1];
Bc = [0; 10];
Cc = [1 0];
Dc = 0;

% --- 教材例 4.14 的原始程式在現行 Octave/MATLAB 已無法執行 ---
%   [A,B] = c2d(Ac,Bc,T)          <- 舊版語法，現行 c2d 只吃 LTI 物件
%   [numz,denz] = ss2tf(A,B,C,D)  <- Octave 的 control 套件沒有 ss2tf
% --- 以下為數學內容相同的修正版 ---
sys_c = ss(Ac, Bc, Cc, Dc);        % 步驟 1：打包成 LTI 物件
Gp2   = tf(sys_c);                 % 步驟 2：類比轉移函數（式 4-76）
sys_d = c2d(sys_c, T, 'zoh');      % 步驟 3：離散化（式 4-69、4-71）
[A, B, C, D] = ssdata(sys_d);      % 取出離散矩陣
Gz2   = tf(sys_d);                 % 步驟 4：脈衝轉移函數（式 4-78）

printf('離散 A =\n'); disp(A);
printf('離散 B =\n'); disp(B);
printf('教材例 4.13：A = [1 0.0952; 0 0.905]，B = [0.0484; 0.952]\n');
printf('C = [%g %g]（= Cc，未改變），D = %g（= Dc，未改變）\n\n', C, D);

[n6, d6] = tfdata(Gz2, 'v');
printf('脈衝轉移函數 G(z)：\n');
printf('  分子 = [%.6g %.6g %.6g]\n', n6);
printf('  分母 = [%.6g %.6g %.6g]\n', d6);

### 結果解讀

`c2d` 算出的 $A$、$B$ 與教材例 4.13 的手算結果**完全吻合**（教材取三位有效數字）。

$C$、$D$ 也如公式所述**完全沒有改變**——這印證了「離散化只動 $A$ 和 $B$」。

> **與 Ackermann 範例的關係**：這裡的 $A=e^{A_cT}$、$B=\left[\int_0^Te^{A_c\nu}d\nu\right]B_c$，就是 [`course/ackermann`](../ackermann/full-Ackermann-formula-example.md) 步驟 2 用的同一組公式，也是 `c2d(sys,T,'zoh')` 在背後做的事。
>
> **差別只在 $A_c$ 的右下角**：本例是 $-1$（有摩擦），衛星例子是 $0$（太空中無摩擦）。這一格之差，就是「馬達會自己停下來」與「衛星永遠停不下來」的分野。

---


## 八、級數展開法需要幾項？(式 4-72、4-74)

### 要驗證什麼

教材說「三位有效數字需 3 項，六位有效數字需 5 項」。下一格用程式把級數逐項累加，跟 `c2d` 的結果比對，看誤差怎麼收斂。

### ⚠️ 這一格有本章最致命的語法陷阱

$$\Phi_c(T)=\sum_{k=0}^{\infty}\frac{A_c^kT^k}{k!}$$

其中 $A_c^k$ 是**矩陣次方**（$A_c$ 自乘 $k$ 次），程式要寫 `Ac^k`。

**若寫成 `Ac.^k`（加點），Octave 會對每個元素分別取次方——答案完全錯誤，而且不會報錯。**

| 數學 | 正確 | 錯誤 |
|---|---|---|
| $A_c^k$（矩陣次方） | `Ac^k` | ~~`Ac.^k`~~ |
| $e^{A_cT}$（矩陣指數） | `expm(Ac*T)` | ~~`exp(Ac*T)`~~ |
| $k!$ | `factorial(k)` | — |

**同理 `expm` 與 `exp`**：`expm([0 1;0 0])` 得到 $\begin{bmatrix}1&1\\0&1\end{bmatrix}$，而 `exp([0 1;0 0])` 得到 $\begin{bmatrix}1&e\\1&1\end{bmatrix}$——**完全不同的東西**。


In [ ]:
%% 實驗 7：級數展開法 vs c2d（式 4-72、4-74）
printf('  項數   max|A_series - A_c2d|   max|B_series - B_c2d|\n');
for nterms = 1:6
    Phi = zeros(2); Int = zeros(2);
    for k = 0:nterms-1
        Phi = Phi + (Ac^k) * T^k     / factorial(k);       % 式 (4-72)
        Int = Int + (Ac^k) * T^(k+1) / factorial(k+1);     % 式 (4-73)
    end
    Bs = Int * Bc;                                          % 式 (4-74)
    printf('  %-6d %-23.2e %.2e\n', nterms, max(max(abs(Phi-A))), max(abs(Bs-B)));
end

printf('\n順便看一下 expm 與 exp 的差別（本章最致命的陷阱）：\n');
M = [0 1; 0 0];
printf('  expm([0 1;0 0]) =\n'); disp(expm(M));
printf('  exp([0 1;0 0])  =\n'); disp(exp(M));
printf('  => 完全不同！矩陣指數必須用 expm\n');

### 結果解讀

誤差每加一項大約**縮小 30 倍**（因為 $T=0.1$，級數的下一項大約是前一項的 $T/k$ 倍）：

| 項數 | $A$ 的誤差 | 大約幾位有效數字 |
|---|---|---|
| 1 | $9.5\times10^{-2}$ | 1 位 |
| 3 | $1.6\times10^{-4}$ | **3～4 位**（教材說 3 項） |
| 5 | $8.2\times10^{-8}$ | **7 位**（教材說 6 位需 5 項） |

**與教材的說法完全吻合。**

`expm` vs `exp` 的對照也一目了然：

```text
expm([0 1; 0 0]) = [1 1]        <- 矩陣指數（正確）
                   [0 1]

exp([0 1; 0 0])  = [1 e]        <- 逐元素指數（錯誤）
                   [1 1]
```

> **⚠️ 實務警告（教材原文）**：對某些系統，**直接照式 (4-74)、(4-77) 寫程式會產生無法接受的數值誤差**，必須改用其他演算法。
>
> **這也是為什麼實務上直接用 `c2d()`**——它內部採用數值穩定的演算法（如 scaling-and-squaring 的矩陣指數），而不是天真的級數截斷。**級數展開適合用來「理解公式」，不適合用來「做生產計算」。**

---


## 九、本章總結

### 公式速查表

| 式號 | 公式 | 用途 | 對應實驗 |
|---|---|---|---|
| (4-3) | $E(z)=E^*(s)\big|_{e^{sT}=z}$ | z 轉換與星號轉換的橋樑 | — |
| (4-9) | $C(z)=G(z)E(z)$ | **脈衝轉移函數**（本章核心） | 1 |
| (4-14) | $\lim_{z\to1}G(z)=\lim_{s\to0}G_p(s)$ | **直流增益檢查** | 2 |
| (4-17) | $C(z)=G_1(z)G_2(z)E(z)$ | 中間**有**取樣器 | 3 |
| (4-18) | $C(z)=\overline{G_1G_2}(z)E(z)$ | 中間**沒有**取樣器 | 3 |
| (4-19) | $\overline{G_1G_2}(z)\neq G_1(z)G_2(z)$ | **最容易犯的錯** | 3 |
| (4-24) | $C(z)=G(z)D(z)E(z)$ | 含數位濾波器 | 4 |
| (4-27) | $E(z,m)=E(z,\Delta)\big|_{\Delta=1-m}$ | **修正 z 轉換** | 5 |
| (4-39) | $C(z)=z^{-k}G(z,m)E(z)$ | 含時間延遲 | 5 |
| (4-51) | 非同步取樣 | 兩取樣器不同步 | — |
| (4-69) | $A=e^{A_cT}$ | 離散化系統矩陣 | 6, 7 |
| (4-71) | $B=\left[\int_0^T\Phi_c(\nu)d\nu\right]B_c$ | 離散化輸入矩陣 | 6, 7 |
| (4-70) | $C=C_c,\ D=D_c$ | 量測關係不變 | 6 |
| (4-78) | $G(z)=C[zI-A]^{-1}B+D$ | 由狀態模型求 $G(z)$ | 6 |

### MATLAB / Octave 常見錯誤

| 錯誤 | 症狀 | 正確做法 |
|---|---|---|
| **`exp(Ac*T)` 當矩陣指數** | 不報錯，但答案完全錯 | **`expm(Ac*T)`** |
| **`Ac.^k` 當矩陣次方** | 不報錯，但答案完全錯 | **`Ac^k`** |
| `tf(num,den)` 忘了第三引數 `T` | 被當成連續系統，`dcgain` 代 $s=0$ 而非 $z=1$ | `tf(num,den,T)` |
| `c2d(Ac,Bc,T)` 傳裸矩陣 | `'c2d' undefined` | 先 `ss(Ac,Bc,Cc,Dc)` 打包 |
| 用 `ss2tf` | Octave 沒有這個函數 | `tf(sys)` |
| `filter` 分子沒補零 | 序列整體提前一步 | `[0, num]` |
| 用 `impulse` 取反 z 轉換 | Octave 會乘上 $1/T$ | 用 `filter`，或把結果乘 `T` |
| 先離散化再相乘 vs 先相乘再離散化 | 得到不同系統（式 4-19） | 看方塊圖中間**有沒有取樣器** |

### 承先啟後

本章完成了開迴路離散系統的完整分析工具：

1. **$E(z)=E^*(s)\big|_{e^{sT}=z}$** — 把第 3 章的星號轉換翻譯成好用的 z 轉換
2. **$C(z)=G(z)E(z)$** — 解決了「取樣器沒有轉移函數」的難題
3. **修正 z 轉換** — 處理非整數倍延遲，也讓我們看得到取樣點之間
4. **$A=e^{A_cT}$、$B=\int\Phi_c B_c$** — 直接從連續狀態模型得到離散狀態模型，保留自然狀態

**第 5 章**會把這些結果延伸到**閉迴路系統**——也就是把輸出接回輸入端。而第 9 章的**極點安置與狀態估測**，用的就是本章第 4.10 節建立的離散狀態模型（見 [`course/ackermann`](../ackermann/full-Ackermann-formula-example.md) 的完整範例）。
